# Day 7 — Strings, regex, pathlib
Objectives:
- Clean and format text.
- Use regex for parsing.
- Work with file paths using pathlib.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-07`. Read
`python/ds-60day/companion-guides/day07_strings_regex_pathlib.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Strings are immutable sequences of Unicode characters. Ordinary string
methods are the clearest tool for fixed separators and normalization.
A regular expression describes a family of text shapes; it should be
used when the format truly varies according to a pattern, not as a
default replacement for simple string operations.

A filesystem path is structured data, not just a string with slashes.
`pathlib.Path` joins components and exposes names, stems, suffixes, and
parent folders using the host operating system's rules. Plan and
validate a rename before mutating files, preserve suffixes, and detect
collisions.

### Vocabulary

- **Unicode:** the character system Python strings represent.
- **normalization:** turning equivalent input forms into one chosen representation.
- **regular expression:** a pattern language for matching bounded text formats.
- **capture group:** a named or numbered subpart retained from a regex match.
- **path component:** one structured folder or filename element.
- **suffix:** a final filename extension such as `.csv`.

## Syntax anatomy

`pattern.fullmatch(text)` requires the entire input to match, while
`search` can find a match inside longer text. Raw string syntax such as
`r"\d{4}"` preserves backslashes for the regex engine. In
`folder / "report.csv"`, `/` is `Path` joining syntax, not division.

### Worked example 1 — Extract named fields from a bounded filename

Use `fullmatch` when extra text must be rejected. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import re

pattern = re.compile(
    r"sales_(?P<date>\d{4}-\d{2}-\d{2})_(?P<region>[a-z]+)\.csv"
)
match = pattern.fullmatch("sales_2025-07-01_west.csv")
match.groupdict() if match else None

**Expected observation:** `{'date': '2025-07-01', 'region': 'west'}`. A filename with extra trailing text returns `None`.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Build a path without platform-specific separators

Let `Path` own path joining and filename fields. Predict first; then run the next cell.

In [ ]:
from pathlib import Path

report = Path("artifacts") / "daily sales.csv"
(report.parent, report.stem, report.suffix, report.with_suffix(".json"))

**Expected observation:** A tuple of `Path` values is displayed. On every supported operating system the components remain meaningful without hard-coded `/` or `\` separators.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Use `repr(text)` when whitespace, escapes, or invisible characters affect a match.
2. Choose `fullmatch`, `match`, or `search` deliberately and print `match.groupdict()` during diagnosis.
3. Use a raw string for regex patterns containing backslashes.
4. Calculate every destination and collision before calling `rename`.

**Alternative to compare:** Prefer `split`, `partition`, `startswith`, and `endswith` for fixed formats; use a parser rather than regex for languages such as HTML.

**Boundary to test:** Mixed case, Unicode, multiple suffixes, existing destinations, and two source names that normalize to the same target need explicit policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import re
from pathlib import Path

text = 'Order #1234 placed by alice@example.com on 2025-01-03'
order_id = re.search(r'#(\d+)', text).group(1)
email = re.search(r'[\w.%-]+@[\w.-]+', text).group(0)
date = re.search(r'\d{4}-\d{2}-\d{2}', text).group(0)
order_id, email, date

# Pathlib demo
artifact_dir = Path('artifacts/day07')
artifact_dir.mkdir(parents=True, exist_ok=True)
(artifact_dir / 'sample.txt').write_text('hello', encoding='utf-8')
list(artifact_dir.glob('*.txt'))


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. From the supplied multiline text, extract email-shaped values, normalize them to lowercase, and return first-seen unique addresses. **Constraints:** use one bounded regex for extraction, then separate normalization and de-duplication steps; do not attempt full Internet-email validation.
   **Verify:** assert differently cased duplicates collapse to one lowercase address in first-seen order and a no-match string returns `[]`.

2. Write `plan_kebab_renames(folder: Path)` that returns source/destination pairs for regular files such as `Quarterly Report.CSV` without renaming them. **Rules:** normalize the stem to lowercase hyphen-separated words, preserve the suffix, skip unchanged names, and reject collisions including case-normalized collisions.
   **Verify:** use a temporary directory and inspect the complete plan before implementing a separate apply step.

### Additional mastery practice

Use ordinary string operations for fixed syntax, regular expressions for patterns, and `pathlib` for portable path semantics.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Compare `"\n"` with `r"\n"` and predict their lengths and printed representations.
   **Progressive hint:** A raw string preserves the backslash needed by many regex patterns.
   **Verify:** Record `len`, `repr`, and printed behavior for both strings; confirm the newline has length 1 and the raw backslash-n has length 2.
4. **Tracing:** Trace named regex groups while parsing `order-2048.csv`; distinguish `group(0)` from the named capture.
   **Progressive hint:** The whole match and captured subparts are different values.
   **Verify:** Assert `group(0)` is `'order-2048.csv'` while the named ID capture is `'2048'`; add a nonmatching filename returning no match.
5. **Implementation:** Implement `parse_report_name(Path)` returning a date and region for names like `sales_2025-07-01_west.csv`, rejecting mismatches.
   **Progressive hint:** Use `fullmatch` so extra suffix text cannot pass silently.
   **Verify:** Assert the valid filename returns the stated date/region and near misses with trailing text, bad date shape, or wrong suffix are rejected.
6. **Debugging:** Repair a greedy `<.*>` pattern that consumes multiple tags in one line, then explain why a real HTML parser is safer for HTML.
   **Progressive hint:** Use a constrained or non-greedy pattern only for a bounded format.
   **Verify:** Demonstrate the greedy overmatch, then assert the bounded repair returns separate intended tags; state why the fixture is not a general HTML parser.
7. **Edge case and explanation:** Build a rename plan to kebab-case that detects collisions before changing any files, including `A B.txt` and `a-b.txt`.
   **Progressive hint:** Separate planning/validation from filesystem mutation.
   **Verify:** Generate a plan containing both colliding names and assert validation raises before any directory entry is renamed.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** From the supplied multiline text, extract email-shaped values, normalize them to lowercase, and return first-seen unique addresses. **Constraints:** use one bounded regex for extraction, then separate normalization and de-duplication steps; do not attempt full Internet-email validation. **Verify:** assert differently cased duplicates collapse to one lowercase address in first-seen order and a no-match string returns `[]`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: From the supplied multiline text, extract email-shaped values, normalize them to lowercase, and return first-seen unique addresses. use one bounded regex for extraction, then se...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Write `plan_kebab_renames(folder: Path)` that returns source/destination pairs for regular files such as `Quarterly Report.CSV` without renaming them. **Rules:** normalize the stem to lowercase hyphen-separated words, preserve the suffix, skip unchanged names, and reject collisions including case-normalized collisions. **Verify:** use a temporary directory and inspect the complete plan before implementing a separate apply step.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Write `plan_kebab_renames(folder: Path)` that returns source/destination pairs for regular files such as `Quarterly Report.CSV` without renaming them. normalize the stem to lowe...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Compare `"\n"` with `r"\n"` and predict their lengths and printed representations. **Progressive hint:** A raw string preserves the backslash needed by many regex patterns. **Verify:** Record `len`, `repr`, and printed behavior for both strings; confirm the newline has length 1 and the raw backslash-n has length 2.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Compare `"\n"` with `r"\n"` and predict their lengths and printed representations. A raw string preserves the backslash needed by many regex patterns. Record `len`, `repr`, and...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace named regex groups while parsing `order-2048.csv`; distinguish `group(0)` from the named capture. **Progressive hint:** The whole match and captured subparts are different values. **Verify:** Assert `group(0)` is `'order-2048.csv'` while the named ID capture is `'2048'`; add a nonmatching filename returning no match.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace named regex groups while parsing `order-2048.csv`; distinguish `group(0)` from the named capture. The whole match and captured subparts are different values. Assert `group...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `parse_report_name(Path)` returning a date and region for names like `sales_2025-07-01_west.csv`, rejecting mismatches. **Progressive hint:** Use `fullmatch` so extra suffix text cannot pass silently. **Verify:** Assert the valid filename returns the stated date/region and near misses with trailing text, bad date shape, or wrong suffix are rejected.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement `parse_report_name(Path)` returning a date and region for names like `sales_2025-07-01_west.csv`, rejecting mismatches. Use `fullmatch` so extra suffix text cannot pas...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a greedy `<.*>` pattern that consumes multiple tags in one line, then explain why a real HTML parser is safer for HTML. **Progressive hint:** Use a constrained or non-greedy pattern only for a bounded format. **Verify:** Demonstrate the greedy overmatch, then assert the bounded repair returns separate intended tags; state why the fixture is not a general HTML parser.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a greedy `<.*>` pattern that consumes multiple tags in one line, then explain why a real HTML parser is safer for HTML. Use a constrained or non-greedy pattern only for a...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Build a rename plan to kebab-case that detects collisions before changing any files, including `A B.txt` and `a-b.txt`. **Progressive hint:** Separate planning/validation from filesystem mutation. **Verify:** Generate a plan containing both colliding names and assert validation raises before any directory entry is renamed.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Build a rename plan to kebab-case that detects collisions before changing any files, including `A B.txt` and `a-b.txt`. Separate planning/validation from filesystem mutation. Ge...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
